A notebook to fine-tune BERT based models using [SpanMarkerNER library](https://github.com/tomaarsen/SpanMarkerNER?tab=readme-ov-file).

## Conda env install

```shell
conda create -n spanmarker_finetune_2 python=3.10     
conda activate spanmarker_finetune_2
conda install pip
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
pip install datasets==3.0.0
pip install transformers<=4.50.0
pip install span_marker
conda install -n spanmarker_finetune_2 ipykernel --update-deps --force-reinstall
```

## Fine-Tuning

The dataset is prepared using: [labelstudio_parsing](labelstudio_parsing.ipynb).

In [8]:
# from pathlib import Path
# from datasets import load_dataset
# from transformers import TrainingArguments
# from span_marker import SpanMarkerModel, Trainer, SpanMarkerModelCardData
# import torch
# from transformers import AutoConfig

# dataset_id = "DFKI-SLT/few-nerd"
# dataset_name = "FewNERD"
# dataset = load_dataset(dataset_id, "supervised")
# dataset = dataset.remove_columns("ner_tags")
# dataset = dataset.rename_column("fine_ner_tags", "ner_tags")
# labels = dataset["train"].features["ner_tags"].feature.names

# dataset_id1 = "P0L3/CliReNER_v_0_0_26"
# dataset_name1 = "CliReNER_v_0_0_26"
# dataset1 = load_dataset(dataset_id1)
# labels1 = dataset1["train"].features["ner_tags"].feature.names

# print(dataset)
# print(dataset1)
# print(10*"---")
# print(dataset["train"][0])
# print(dataset1["train"][0])
# print(10*"---")
# print(labels)
# print(labels1)


In [1]:
from pathlib import Path
from datasets import load_dataset
from transformers import TrainingArguments
from span_marker import SpanMarkerModel, Trainer, SpanMarkerModelCardData
import torch
from transformers import AutoConfig


def main() -> None:
    # Add this check at the beginning
    if torch.cuda.is_available():
        print(f"CUDA is available. Using {torch.cuda.device_count()} GPU(s).")
        print(f"Device Name: {torch.cuda.get_device_name(0)}")
    else:
        print("CUDA is not available. Training will run on CPU.")

    dataset_id = "P0L3/CliReNER_v_1_0_28"
    dataset_name = "CliReNER_v_1_0_28"
    dataset = load_dataset(dataset_id)
    labels = dataset["train"].features["ner_tags"].feature.names
    # ['O', 'art-broadcastprogram', 'art-film', 'art-music', 'art-other', ...

    # Initialize a SpanMarker model using a pretrained BERT-style encoder
    encoder_id = "P0L3/clirebert_clirevocab_uncased"
    model_name = encoder_id.split("/")
    model_id = f"P0L3/span-marker-{encoder_id}-{dataset_name}_25612814_100"
    model = SpanMarkerModel.from_pretrained(
        encoder_id,
        labels=labels,
        # SpanMarker hyperparameters:
        model_max_length=256,
        marker_max_length=128,
        entity_max_length=14,
        # Model card arguments
        model_card_data=SpanMarkerModelCardData(
            model_id=model_id,
            encoder_id=encoder_id,
            dataset_name=dataset_name,
            dataset_id=dataset_id,
            license="cc-by-sa-4.0",
            language="en",
        ),
    )

    if not hasattr(model.config, "encoder") or model.config.encoder is None:
        model.config.encoder = AutoConfig.from_pretrained(encoder_id)

    # Prepare the 🤗 transformers training arguments
    output_dir = Path("models") / model_id
    args = TrainingArguments(
        output_dir=output_dir,
        # Training Hyperparameters:
        learning_rate=5e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=100,
        weight_decay=0.01,
        warmup_ratio=0.1,
        fp16=True,  # Replace `bf16` with `fp16` if your hardware can't use bf16.
        # Other Training parameters
        logging_first_step=True,
        logging_steps=50,
        evaluation_strategy="steps",
        save_strategy="steps",
        eval_steps=3000,
        save_total_limit=5,
        dataloader_num_workers=2,
    )

    # Initialize the trainer using our model, training args & dataset, and train
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
    )
    trainer.train()

    # Compute & save the metrics on the test set
    metrics = trainer.evaluate(dataset["test"], metric_key_prefix="test")
    trainer.save_metrics("test", metrics)

    # Save the final checkpoint
    trainer.save_model(output_dir / "checkpoint-final")

if __name__ == "__main__":

    
    main()

c:\Users\Desktop\.conda\envs\spanmarker_finetune_2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA is available. Using 1 GPU(s).
Device Name: NVIDIA GeForce RTX 4070 Ti


c:\Users\Desktop\.conda\envs\spanmarker_finetune_2\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Desktop\.cache\huggingface\hub\datasets--P0L3--CliReNER_v_1_0_28. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating test split: 100%|██████████| 120/120 [00:00<00:00, 59982.90 examples/s]
The prov

Step,Training Loss,Validation Loss,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
3000,0.002200,0.071672,0.615099,0.609816,0.612446,0.824106
6000,0.001100,0.085625,0.646214,0.607362,0.626186,0.827020
9000,0.000400,0.092625,0.627728,0.600000,0.613551,0.828344


Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Spreading data between multiple samples: 100%|██████████| 114/114 [00:00<00:00, 893.94 examples/s]
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Spreading data between multiple samples: 100%|██████████| 120/120 [00:00<00:00, 918.72 examples/s] 


## Inference

In [1]:
from span_marker import SpanMarkerModel
import re
model = SpanMarkerModel.from_pretrained("C:\\Users\\ANDRIJA_RAD\\CWED4ETA\\CWED4ETA\\SPANMARKER\\models\\P0L3\\span-marker-P0L3\\clirebert_clirevocab_uncased-CliReNER_v_0_0_26_25612814_100\\checkpoint-final")


c:\Users\Desktop\.conda\envs\spanmarker_finetune_2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
SpanMarker model predictions are being computed on the CPU while CUDA is available. Moving the model to CUDA using `model.cuda()` before performing predictions is heavily recommended to significantly boost prediction speeds.


In [9]:
text =  [
    "Sea levels respond to climate change on timescales from decades to millennia.",
    "This additional commitment would grow to 0.8 m (0.5–1.4 m) for emissions until 2090, of which 0.6 m (0.4–1.1 m) could be avoided under very stringent mitigation.",
    "Sea levels respond to GHG emissions and global warming on timescales from decades to millennia.",
    "Residents in US cities annually consume 11.1 Mt of meat: 4.6 Mt of chicken, 3.7 Mt of beef and 2.7 Mt of pork.",
    "Mountain glaciers play a key role in climate–land interactions, and shape the mountain water cycle1, influence mountain hazards2 and form an essential part of the world’s water towers that meet the needs of billions of people globally3. ",
    "At colder mean equivalent temperatures, TaGla is strongly coupled to TaAmb with k values close to unity."
]

entities_list = model.predict(text)

for i, entities in enumerate(entities_list):
    print("Sentence: ", text[i])
    for entity in entities:
        print(entity["span"], "=>", entity["label"])
    print()


SpanMarker model predictions are being computed on the CPU while CUDA is available. Moving the model to CUDA using `model.cuda()` before performing predictions is heavily recommended to significantly boost prediction speeds.


Sentence:  Sea levels respond to climate change on timescales from decades to millennia.
Sea levels => Quantity
climate change => Meteorological Phenomenon
timescales => Time Period
millennia => Time Period

Sentence:  This additional commitment would grow to 0.8 m (0.5–1.4 m) for emissions until 2090, of which 0.6 m (0.4–1.1 m) could be avoided under very stringent mitigation.
0.8 m => Quantity
0.5–1.4 m => Quantity
emissions => Physical Phenomenon
2090 => Time Period
0.6 m => Quantity
0.4–1.1 m => Quantity
very stringent mitigation => Other

Sentence:  Sea levels respond to GHG emissions and global warming on timescales from decades to millennia.
Sea levels => Quantity
GHG emissions => Physical Phenomenon
global warming => Quantity
timescales => Time Period
millennia => Time Period

Sentence:  Residents in US cities annually consume 11.1 Mt of meat: 4.6 Mt of chicken, 3.7 Mt of beef and 2.7 Mt of pork.
Residents => Person
US cities => Location
11.1 Mt => Quantity
meat => Chemical
4.6